# lme4 (mixed models)

A refresher on **lme4** — the de-facto R package for fitting **linear and generalized linear mixed-effects models** (LMMs / GLMMs). When your data has *grouping structure* — repeated measures per subject, students nested in schools, items in a psycholinguistics experiment — ordinary regression's "every row is independent" assumption breaks. lme4's `lmer()` / `glmer()` let you add **random effects** that model that structure directly.

**Domain:** Data Analysis & Research  ·  **from study list**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _R (lme4 package)_

## 1. What & Why

**What it is.** `lme4` is an R package (Bates, Mächler, Bolker & Walker) for fitting **mixed-effects models** — regression models that contain both **fixed effects** (population-level coefficients you want to estimate, like a treatment effect) and **random effects** (group-level deviations drawn from a distribution, like a per-subject intercept). The workhorses are `lmer()` for Gaussian responses and `glmer()` for binomial/Poisson/etc. It fits by (restricted) maximum likelihood using a fast sparse-matrix, penalized-least-squares engine.

**The problem it solves.** Most real datasets are **not** independent rows. You measure the same 30 subjects 10 times each; you sample students from 20 classrooms; you show 50 words to every participant. Ordinary least squares treats all 300 rows as independent, **underestimates standard errors**, and inflates false positives. The two naive fixes are both bad: *complete pooling* (ignore groups) throws away real structure; *no pooling* (a separate fit per group, or a fixed-effect dummy per group) overfits small groups and can't generalize to new groups. Mixed models give you **partial pooling** — each group's estimate is shrunk toward the population mean by an amount that depends on how noisy and how large that group is. You get correct standard errors *and* a principled compromise between the two extremes.

**When to reach for it.** Repeated-measures / longitudinal data, hierarchical/nested data (pupils in schools in districts), crossed designs (subjects × items in psycholinguistics), and any time you'd otherwise be tempted to run a separate regression per group. **When not to:** if there's no grouping structure, a plain `lm()`/`glm()` is simpler and correct; if you have very few groups (< ~5–6 levels), a random effect can't estimate its variance well — use a fixed effect instead.

## 2. Mental Model

**Fixed effects draw one line for everyone; random effects let each group wobble around that line, and the wobble is itself estimated and reined in.**

Picture a scatter of reaction-time-vs-day for 18 subjects:

- **Complete pooling** (`lm(y ~ x)`): one regression line for the whole cloud. Ignores that subject 308 is slow and subject 309 is fast.
- **No pooling** (a line fit independently per subject): 18 separate, unconstrained lines. The line for a subject with only 2 noisy points is garbage.
- **Partial pooling** (`lmer(y ~ x + (x | subject))`): one *population* line (the fixed effects) plus a per-subject intercept and slope drawn from a normal distribution whose variance the model estimates. Each subject's line is pulled toward the population line — **a lot** if that subject has little/noisy data, **barely** if they have lots of clean data. This automatic, data-driven **shrinkage** is the whole point.

The formula syntax encodes this: everything outside parentheses is fixed (population-level); everything inside `( ... | group )` is a random effect that varies by `group`. `(1 | g)` = random intercept; `(1 + x | g)` = random intercept *and* slope (correlated by default).

## 3. Key Concepts

- **Fixed vs random effects.** *Fixed* = the levels you care about and want to estimate directly (drug vs placebo). *Random* = levels you regard as a sample from a larger population and only want to account for (these particular 30 subjects). Rule of thumb: if you'd want the same coefficient if you re-ran with *different* subjects, it's random.
- **Random intercept `(1 | g)`.** Each group gets its own baseline, `b_g ~ N(0, σ²_g)`. Models "some subjects are just slower."
- **Random slope `(1 + x | g)` or `(x | g)`.** Each group gets its own *effect of x* too. Models "the practice effect is stronger for some subjects." By default lme4 also estimates the **correlation** between the random intercept and slope.
- **`(x || g)`** — uncorrelated random intercept and slope (forces the intercept-slope correlation to 0). Useful for simpler, faster-converging models.
- **Crossed vs nested.** *Nested*: each classroom belongs to exactly one school → `(1 | school/classroom)`. *Crossed*: every subject sees every item → `(1 | subject) + (1 | item)`. lme4 handles crossed effects natively and efficiently (a big reason it beat older tools).
- **REML vs ML.** `lmer` defaults to **REML** (less-biased variance-component estimates) — best for reporting. But you **must refit with `REML = FALSE`** (i.e. ML) to compare models that differ in *fixed* effects via likelihood-ratio tests or AIC.
- **Shrinkage / partial pooling.** The BLUPs (conditional modes) for each group are shrunk toward 0; small/noisy groups shrink more.
- **Variance components.** The output's headline numbers: how much variance lives between groups vs residual. `VarCorr(m)` and the random-effects table report these.
- **Singular fit.** When an estimated variance hits 0 or a correlation hits ±1 — the random-effects structure is too complex for the data.
- **No p-values by default.** lme4 deliberately omits p-values for fixed effects (the denominator degrees of freedom are not well-defined). Use `lmerTest`, `confint()`, or likelihood-ratio tests instead.

## 4. Setup

lme4 is an **R package**, not a Python library — there is no `pip install`. You install it from CRAN inside R. (This notebook is therefore *conceptual*: the snippets below run in an R session, not this Python kernel.)

```bash
# Install R itself first (the interpreter):
brew install --cask r            # macOS;  or download from https://cran.r-project.org
sudo apt-get install r-base      # Debian/Ubuntu

# Verify:
R --version
```

```r
# --- inside an R session / Rscript ---
install.packages("lme4")         # the core package
install.packages("lmerTest")     # adds p-values (Satterthwaite df) to lmer output
install.packages("performance")  # easy R^2, ICC, diagnostics for mixed models
# Optional companions:
install.packages(c("broom.mixed", "emmeans", "DHARMa", "ggeffects"))

library(lme4)
packageVersion("lme4")           # e.g. '1.1.35.x'
```

To run R from a Jupyter notebook you'd install the **IRkernel** (`install.packages("IRkernel"); IRkernel::installspec()`); to call it from Python use **`rpy2`** or **pymer4**. The example datasets used below (`sleepstudy`, `cbpp`) ship *with* lme4 — no download needed.

## 5. Worked Examples

**These run in an R session, not this Python kernel.** Paste them into the R REPL, save as `model.R` and run `Rscript model.R`, or use an R Jupyter kernel. Expected results are described in prose — there is no fabricated cell output.

### Example 1 — Random intercepts & slopes (`lmer`, the `sleepstudy` classic)

`sleepstudy`: reaction time (`Reaction`) of 18 subjects over 10 days of sleep deprivation (`Days`). Reaction time worsens with days, but the *baseline* and the *rate* differ per subject — the textbook case for a random slope.

```r
library(lme4)

# Fixed effect of Days (the population trend) + per-Subject intercept AND slope.
m <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy)
summary(m)
```

What to read in `summary(m)`:

- **Fixed effects:** `(Intercept)` ≈ 251 ms, `Days` ≈ 10.5 ms/day — the *average* subject slows ~10 ms per day of deprivation.
- **Random effects:** the `Subject` variance for `(Intercept)` (~612, i.e. SD ~24.7 ms in baseline) and for `Days` (~35, SD ~5.9 ms/day in slope), plus their correlation (~0.07). `Residual` is within-subject noise.
- **No p-value column** — by design. Add it with `lmerTest`:

```r
library(lmerTest)                       # mask lme4::lmer, adds Satterthwaite df + p-values
m  <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy)
summary(m)                              # now the Days row shows t, df, Pr(>|t|)
confint(m, method = "Wald")             # CIs for fixed effects (profile = slower, better)

ranef(m)$Subject                        # per-subject deviations (BLUPs) from the average line
coef(m)$Subject                         # per-subject intercept+slope = fixed + random
fixef(m)                                # just the population coefficients
VarCorr(m)                              # variance components as SDs + correlations
```

### Example 2 — Comparing random-effects structures (LRT, ML refit)

Does the random *slope* earn its keep, or is a random intercept enough? Compare with a likelihood-ratio test. **`anova()` on `lmer` objects automatically refits both with ML** (`REML = FALSE`), which is required when comparing — here the models differ only in random structure, so it's also valid under REML, but `anova()` does the safe thing.

```r
m_int   <- lmer(Reaction ~ Days + (1 | Subject),    data = sleepstudy)  # intercept only
m_slope <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy)  # + random slope

anova(m_int, m_slope)   # LRT: Chisq, Df, Pr(>Chisq). Small p => keep the random slope.
```

The random slope is strongly justified here (subjects differ in *how fast* they degrade), so prefer `m_slope`. **To compare models differing in FIXED effects, you must use ML**, not REML:

```r
a <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy, REML = FALSE)
b <- lmer(Reaction ~ 1    + (Days | Subject), data = sleepstudy, REML = FALSE)  # drop Days
anova(a, b)             # tests the fixed effect of Days
```

### Example 3 — Logistic GLMM (`glmer`, binomial `cbpp`)

`cbpp`: contagious bovine pleuropneumonia — `incidence` cases out of `size` cattle, across `herd` and `period`. The response is a proportion → **binomial GLMM** with a per-herd random intercept.

```r
library(lme4)

# cbind(successes, failures) is the binomial response; (1 | herd) = herd-level intercept.
g <- glmer(cbind(incidence, size - incidence) ~ period + (1 | herd),
           data   = cbpp,
           family = binomial)
summary(g)

exp(fixef(g))           # odds ratios for each period vs the reference
```

Notes specific to `glmer`:

- Coefficients are on the **link scale** (log-odds); exponentiate for odds ratios.
- It uses **Laplace approximation** by default; bump accuracy with `nAGQ = 10` (adaptive Gauss-Hermite quadrature) for a single scalar random effect.
- Convergence is touchier than `lmer`. If it warns, try `control = glmerControl(optimizer = "bobyqa")` and **scale/center continuous predictors** first.

### Example 4 — Nested vs crossed, and predictions

```r
# NESTED: classroom is only meaningful within its school.
lmer(score ~ ses + (1 | school/classroom), data = pupils)
#   ^ expands to (1 | school) + (1 | school:classroom)

# CROSSED: every subject responds to every item (psycholinguistics).
lmer(rt ~ condition + (1 | subject) + (1 | item), data = lexdec)

# PREDICTIONS:
predict(m, newdata = nd)                       # includes the subject's random effect
predict(m, newdata = nd, re.form = NA)          # population-level only (ignore random effects)

# Quick R^2 (marginal = fixed only; conditional = fixed + random) and ICC:
performance::r2(m)
performance::icc(m)                             # share of variance attributable to grouping
```

`re.form = NA` answers "what does the *average* subject do"; the default answers "what does *this* subject do." The **ICC** (intraclass correlation) — between-group variance ÷ total variance — is the one number that says how much the grouping matters.

## 6. Gotchas & Pitfalls

- **REML vs ML for comparison.** You **cannot** compare models with different *fixed* effects when both were fit with REML — the likelihoods aren't comparable. Refit with `REML = FALSE` (or just trust `anova()`, which refits for you).
- **No p-values is intentional.** Don't go looking for the `Pr(>|t|)` column in base lme4. Load **`lmerTest`** for Satterthwaite/Kenward-Roger df, or use `confint()` / likelihood-ratio tests. Beware that t-as-z and Wald CIs are anticonservative with few groups.
- **"boundary (singular) fit" warning.** A variance estimated at 0 or a correlation at ±1 means your random structure is too rich for the data. Simplify: drop the random slope, use `(x || g)` to kill the correlation, or accept the simpler model. Don't ignore it.
- **Convergence warnings in `glmer`.** Usually unscaled predictors or an over-specified model. **Center and scale** continuous covariates, try `optimizer = "bobyqa"`, increase `nAGQ`, or simplify random effects.
- **Maximal vs parsimonious random structure.** Barr et al. (2013) urged "keep it maximal" (all theoretically justified random slopes); Matuschek et al. (2017) showed that over-parameterized models hurt power and won't converge. In practice: start reasonable, simplify on singular fits.
- **Too few groups.** A random effect needs a handful of levels (rough floor ~5–6) to estimate its variance. With 2–3 groups, use a **fixed** effect instead.
- **Random slope without the corresponding fixed effect.** Almost always a mistake — put the fixed effect in too (`y ~ x + (x | g)`, not `y ~ (x | g)`).
- **`/` is nesting, `:` is interaction.** `(1 | a/b)` ≠ `(1 | a) + (1 | b)`. If your nesting codes aren't unique across groups (e.g. classroom "1" exists in every school), use explicit nesting or unique IDs.
- **Don't `anova()` an `lmer` against an `lm`** to test "is the random effect needed" naively — the test is on the boundary of the parameter space (variance ≥ 0), so the usual χ² reference is conservative; use `RLRsim` or `ranova()` (lmerTest) instead.

## 7. When to Use vs Alternatives

| Option | Best at | Weaknesses vs lme4 |
|---|---|---|
| **lme4 (`lmer`/`glmer`)** | Fast frequentist LMMs/GLMMs, large crossed random effects, the field standard in psych/linguistics/ecology | No p-values out of the box; no easy custom variance structures (heteroscedasticity, AR(1) residuals); GLMM families limited |
| **`nlme` (R)** | Models needing correlated/heteroscedastic *residuals* (AR(1), spatial), nonlinear mixed models | Slow & awkward with **crossed** random effects (its main weakness — the reason lme4 exists) |
| **`glmmTMB` (R)** | Zero-inflation, negative binomial, beta, AR(1) and spatial random effects; fits via Template Model Builder | Smaller community, less tooling than lme4 |
| **Bayesian: `brms` / `rstanarm` (R, Stan)** | Full uncertainty, priors, complex/hierarchical structures, custom likelihoods; genuine credible intervals | Much slower (MCMC); requires priors and convergence checking |
| **`statsmodels` MixedLM (Python)** | Staying in Python; basic random intercepts/slopes | Far less capable: weak/no crossed-effects support, no GLMM, fewer diagnostics |
| **`lm`/`glm`** | No grouping structure at all | Wrong standard errors when data is clustered/repeated |

**Default:** reach for `lme4` for standard frequentist mixed models. Move to **`glmmTMB`** when you need zero-inflation or fancy distributions, **`nlme`** for correlated residuals, and **`brms`** when you want full Bayesian uncertainty or the model is too custom for any of the above.

## 8. Resources

- **lme4 on CRAN** — package home, manual, and vignettes (`vignette("lmer", package = "lme4")`): https://cran.r-project.org/package=lme4
- **Bates, Mächler, Bolker & Walker (2015), "Fitting Linear Mixed-Effects Models Using lme4"** — the canonical paper (also `vignette("lmer")`), *J. Stat. Soft.*: https://www.jstatsoft.org/article/view/v067i01
- **GLMM FAQ** (Ben Bolker) — the single most useful troubleshooting reference for mixed models in R: https://bbolker.github.io/mixedmodels-misc/glmmFAQ.html
- **lmerTest** — p-values and `ranova()` for random effects: https://cran.r-project.org/package=lmerTest
- **Brauer & Curtin / Singmann & Kellen tutorials** and **"An Introduction to Linear Mixed-Effects Modeling in R"** (Brown, 2021, *AMPPS*): https://journals.sagepub.com/doi/full/10.1177/2515245920960351